# Centralized Reward Learning

This notebook trains a model to predict the **immediate centralized reward**:
- Reward = 1.0 if acting agent is on an apple
- Reward = 0.0 otherwise

**Correction:** Fixed trajectory logic to match Algorithm 1 (c_t persists across steps).

In [1]:
# =============================================================================
# PARAMETERS
# =============================================================================

# --- Environment ---
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 1
NUM_APPLES = 5

# --- Model ---
MODEL_TYPE = "CNN"  # "CNN" or "MLP"
MLP_HIDDEN_DIM = 32
MLP_NUM_LAYERS = 2
CNN_CONV_CHANNELS = [4]
CNN_KERNEL_SIZE = 1
CNN_HEAD_HIDDEN_DIM = 8
CNN_HEAD_NUM_LAYERS = 1

# --- Training ---
LR = 0.001
BATCH_SIZE = 64
DISCOUNT = 0.0  # Reward learning
MAX_STEPS = 100000
EVAL_FREQ = 1000

# --- Benchmarks ---
SOFT_THRESHOLD = 1.0  # Mean error < 1%
HARD_THRESHOLD = 1.0  # Max error < 1%
NUM_TEST_SAMPLES = 1000

# --- Output ---
EXPERIMENT_NAME = "reward_cen"
OUTPUT_DIR = "outputs"
CSV_PATH = "outputs/experiment_results.csv"
SEED = 42

In [2]:
import sys
import time
import ast
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
from tqdm import tqdm

sys.path.append("../../")

# Environment & Helpers
from tadd_helpers.env_functions import State, init_empty_state
from tadd_helpers.setting_seed import set_all_seeds
from teleport_dynamic.base_value_model import BaseValueModelV2
from teleport_dynamic.model_cen_3ch import ValueCNNCentralized3Ch, ValueMLPCentralized3Ch
from teleport_dynamic.rewards_centralized import get_reward_centralized
from teleport_dynamic.orchard_generator import init_fixed_apples, teleport_agent
from teleport_dynamic.training_functions import train_reward_from_buffer
from teleport_dynamic.experiment_utils import (
    generate_centralized_test_set,
    evaluate_centralized_model,
    check_benchmark_centralized,
    append_experiment_result,
    format_eval_results_for_log,
    get_current_ram_mb
)

if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_all_seeds(SEED)

# Define Architecture String for Filename
if MODEL_TYPE == "CNN":
    # Format: conv[16,32]_k1_head32x1
    chan_str = str(CNN_CONV_CHANNELS).replace(" ", "")
    arch_str = f"conv{chan_str}_k{CNN_KERNEL_SIZE}_head{CNN_HEAD_HIDDEN_DIM}x{CNN_HEAD_NUM_LAYERS}"
else:
    # Format: mlp64x2
    arch_str = f"mlp{MLP_HIDDEN_DIM}x{MLP_NUM_LAYERS}"

# Construct Descriptive Log Filename
# Example: reward_cen_CNN_6x6_1ag_0-1_conv[16,32]_k1_head32x1.txt
log_filename = f"{EXPERIMENT_NAME}_{MODEL_TYPE}_{WIDTH}x{HEIGHT}_{NUM_AGENTS}ag_0-1_{arch_str}.txt"

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUTPUT_DIR / log_filename

print(f"Logging to: {LOG_FILE}")

KeyboardInterrupt: 

In [ ]:
# Model Initialization
model: BaseValueModelV2

if MODEL_TYPE == "CNN":
    model = ValueCNNCentralized3Ch(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        mlp_hidden_dim=CNN_HEAD_HIDDEN_DIM, mlp_num_layers=CNN_HEAD_NUM_LAYERS,
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE, device=DEVICE
    )
    model_config = f"conv={CNN_CONV_CHANNELS},k={CNN_KERNEL_SIZE},head={CNN_HEAD_HIDDEN_DIM}"
else:
    model = ValueMLPCentralized3Ch(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS, device=DEVICE
    )
    model_config = f"hidden={MLP_HIDDEN_DIM},layers={MLP_NUM_LAYERS}"

num_params = model.get_num_parameters()
print(f"Params: {num_params:,}")

Params: 1,185


In [ ]:
# Initialize State
state = init_fixed_apples(WIDTH, HEIGHT, NUM_AGENTS, NUM_APPLES)

# Generate Test Set
test_sets = generate_centralized_test_set(
    HEIGHT, WIDTH, NUM_AGENTS, NUM_APPLES, NUM_TEST_SAMPLES, state.apples, SEED + 1000
)
print("Test set generated.")

Test set generated.


In [ ]:
with open(LOG_FILE, "w") as f:
    f.write(f"START Experiment: {EXPERIMENT_NAME} | {MODEL_TYPE}\n")
    f.write(f"Grid: {WIDTH}x{HEIGHT}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}\n")
    f.write(f"Params: {num_params:,} | Config: {model_config}\n")
    f.write(f"Benchmarks: Soft < {SOFT_THRESHOLD}%, Hard < {HARD_THRESHOLD}%\n")
    f.write(f"Log Format: Category: Mean Error % / Max Error %\n")
    f.write("="*80 + "\n")

soft_benchmark_step = None
hard_benchmark_step = None
last_loss = 0.0
start_time = time.time()

# Algorithm Line 3: Initialize acting agent c_t randomly *outside* loop
c_t = np.random.randint(0, NUM_AGENTS)

print(f"Starting training for {MAX_STEPS} steps...")

for step in range(MAX_STEPS):
    # 1. Get Reward
    reward = get_reward_centralized(state, c_t)
    
    # 2. Transition
    next_state = state.copy()
    teleport_agent(next_state, c_t)
    
    # 3. Sample NEXT actor
    c_t_next = np.random.randint(0, NUM_AGENTS)
    
    # 4. Add to Buffer & Train
    model.add_experience(state, next_state, reward, c_t, c_t_next)
    loss = train_reward_from_buffer(model, BATCH_SIZE)
    if loss is not None:
        last_loss = loss
    
    # 5. Update State & Actor
    state = next_state
    c_t = c_t_next
    
    # 6. Evaluation
    if step > 0 and step % EVAL_FREQ == 0:
        results = evaluate_centralized_model(model, test_sets)
        soft, hard = check_benchmark_centralized(results, SOFT_THRESHOLD, HARD_THRESHOLD)
        ram_mb = get_current_ram_mb()
        
        if soft and not soft_benchmark_step: soft_benchmark_step = step
        if hard and not hard_benchmark_step: hard_benchmark_step = step
        
        status = "HARD" if hard else ("SOFT" if soft else "-")
        log_line = format_eval_results_for_log(results, step, last_loss)
        full_log = f"{log_line} | RAM: {ram_mb:.1f}MB | Bench: {status}"
        
        with open(LOG_FILE, "a") as f:
            f.write(full_log + "\n")
        
        if hard:
            print(f"Hard benchmark achieved at step {step}!")
            break

wall_time = time.time() - start_time

# Final Evaluation to Log
final_results = evaluate_centralized_model(model, test_sets)
final_mean = max(r.mean_pct_error for r in final_results.values())
final_max = max(r.max_pct_error for r in final_results.values())

with open(LOG_FILE, "a") as f:
    f.write("="*80 + "\n")
    f.write(f"FINISHED. Time: {wall_time:.2f}s\n")
    f.write(f"Soft Step: {soft_benchmark_step}, Hard Step: {hard_benchmark_step}\n")
    f.write(f"Final Mean Err: {final_mean:.4f}%, Final Max Err: {final_max:.4f}%\n")
    f.write(format_eval_results_for_log(final_results, step, last_loss) + "\n")

print(f"Training done. Log saved to {LOG_FILE}")

Starting training for 100000 steps...
Hard benchmark achieved at step 1000!
Training done. Log saved to outputs/reward_cen_CNN_6x6_1ag_0-1_conv[4]_k1_head8x1.txt


In [ ]:
# CSV Appending
append_experiment_result(
    CSV_PATH, 
    "reward", 
    MODEL_TYPE, 
    True,  # centralized
    f"{WIDTH}x{HEIGHT}", 
    NUM_AGENTS, 
    NUM_APPLES, 
    "centralized",
    model_config, 
    soft_benchmark_step, 
    hard_benchmark_step,
    step, 
    final_mean, 
    final_max, 
    wall_time, 
    # NEW:
    num_parameters=model.get_num_parameters(),
    kernel_size=CNN_KERNEL_SIZE if MODEL_TYPE == "CNN" else None,
    learning_method="reward",
    notes=""
)

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK: Visualizing 3 samples per category")
print("="*80)

for category, cases in test_sets.items():
    if not cases: continue
    
    print(f"\n>>> CATEGORY: {category.upper()} <<<")
    # Take first 3
    samples = cases[:3]
    
    for i, case in enumerate(samples):
        print(f"\n--- Sample {i+1} ---")
        print(case.state) # Visualizes grid
        print(f"Acting Agent: {case.acting_agent_idx}")
        
        # Visualize Input
        inp = model.raw_state_to_nn_input(case.state, case.acting_agent_idx)
        print(f"NN Input Shape: {inp.shape}")
        
        if MODEL_TYPE == "CNN":
            # Input is (3, H, W). Print each channel.
            channel_names = ["Apples", "Agents", "Actor"]
            for ch_idx, ch_name in enumerate(channel_names):
                print(f"\nChannel {ch_idx} ({ch_name}):")
                print(inp[ch_idx])
        else:
            # MLP Input is flattened
            print(f"\nInput Vector:\n{inp}")

        # Prediction
        pred = model.get_value(case.state, case.acting_agent_idx)
        actual = case.true_reward
        print(f"\nPREDICTED: {pred:.4f} | ACTUAL: {actual:.4f} | ERROR: {abs(pred-actual):.4f}")


SANITY CHECK: Visualizing 3 samples per category

>>> CATEGORY: PICKER <<<

--- Sample 1 ---
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (5, 5)

--- Agents (Count) ---
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . 1

--- Apples (Count) ---
. . . . 1 .
. . . . . .
. 1 . . . 1
. . . . . .
. . . . . 1
. . . . . 1
Acting Agent: 0
NN Input Shape: (3, 6, 6)

Channel 0 (Apples):
[[0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 1.]]

Channel 1 (Agents):
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1.]]

Channel 2 (Actor):
[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1.]]

PREDICTED: 0.9999 | ACTUAL: 1.0000 | ERROR: 0.0001

--- Sample 2 ---
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (0, 4)

--- Agen